# Lesson 5: Answering Questions over SQL with a Claude Tool-Use Agent

**Migration note:** The original lesson used the OpenAI **Assistants API** (`client.beta.assistants` / threads / runs). Anthropic Claude has **no equivalent of the Assistants API**, so this lesson is rebuilt as a **Claude tool-use agent**: a small agentic loop that lets Claude call the same SQL helper functions (`Helper.tools_sql`) to answer the same natural-language questions over the Covid SQLite database. See MIGRATION-NOTES.md for details.

In [ ]:
import os
import json
import Helper
from anthropic import Anthropic
from Helper import get_positive_cases_for_state_on_date
from Helper import get_hospitalized_increase_for_state_on_date
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment

In [ ]:
INSTRUCTIONS = """You are an assistant answering questions
                  about a Covid dataset."""

available_functions = {
    "get_positive_cases_for_state_on_date": get_positive_cases_for_state_on_date,
    "get_hospitalized_increase_for_state_on_date": get_hospitalized_increase_for_state_on_date,
}


def run_conversation(question):
    """Answer a natural-language question over the SQL database using
    Claude tool use. Loops until Claude stops requesting tools."""
    messages = [{"role": "user", "content": question}]

    while True:
        response = client.messages.create(
            model=os.getenv("LLM_MODEL", "claude-haiku-4-5"),
            max_tokens=2048,
            system=INSTRUCTIONS,
            messages=messages,
            tools=Helper.tools_sql,
        )

        if response.stop_reason != "tool_use":
            # Claude has produced its final answer
            return "".join(
                block.text for block in response.content if block.type == "text"
            )

        # Execute every requested tool and feed the results back
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for tool_call in response.content:
            if tool_call.type != "tool_use":
                continue
            function_to_call = available_functions[tool_call.name]
            function_response = function_to_call(
                state_abbr=tool_call.input.get("state_abbr"),
                specific_date=tool_call.input.get("specific_date"),
            )
            print(f"{tool_call.name} -> {function_response}")
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": tool_call.id,
                    "content": str(function_response),
                }
            )
        messages.append({"role": "user", "content": tool_results})

In [ ]:
answer = run_conversation(
    """how many hospitalized people we had in Alaska
       the 2021-03-05?"""
)
print(answer)

In [ ]:
answer = run_conversation(
    """how many new positive cases did California have
       on 2021-03-05?"""
)
print(answer)